In [33]:
import json
import pandas as pd
import os

# Load JSON
json_path = "/Users/mariem/Desktop/Project_ADA/newyorker_caption_contest/contests.json"
with open(json_path, "r") as f:
    contests = json.load(f)

# Flatten nested metadata

df_contests = pd.DataFrame(contests)


df_contests.iloc[-1]['metadata']={'num_captions': None, 'num_votes': None}
print(df_contests.iloc[-1:]['metadata'])

# Display the first few columns
df_contests.head()


383    None
Name: metadata, dtype: object


/var/folders/sr/7lgvwwwx7051vshyb_tzhm4r0000gn/T/ipykernel_43987/342937404.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_contests.iloc[-1]['metadata']={'num_captions': None, 'num_votes': None}


,contest_id,image,data,metadata
0,510,images/510.jpg,data/510.csv,"{'num_captions': 3905, 'num_votes': 41185, 'im..."
1,511,images/511.jpg,data/511.csv,"{'num_captions': 3325, 'num_votes': 28205, 'im..."
2,512,images/512.jpg,data/512.csv,"{'num_captions': 4399, 'num_votes': 21574, 'im..."
3,513,images/513.jpg,data/513.csv,"{'num_captions': 4141, 'num_votes': 16894, 'im..."
4,514,images/514.jpg,data/514.csv,"{'num_captions': 3951, 'num_votes': 95790, 'im..."


Now we identify the missing metadata 


In [31]:
def missing_meta(meta):
    return (
        not isinstance(meta, dict)
        or ("num_captions" not in meta or meta.get("num_captions") in [None, ""])
        or ("num_votes" not in meta or meta.get("num_votes") in [None, ""])
    )

missing_df = df_contests[df_contests["metadata"].apply(missing_meta)]

print(f"Contests missing metadata: {len(missing_df)}")
missing_df[["contest_id", "metadata"]].head(10)




Contests missing metadata: 144


,contest_id,metadata
16,527,None
52,564,None
76,588,None
85,597,None
93,605,None
103,615,None
132,644,None
135,647,None
144,656,None
145,657,None


Now let's add the missing metadata

In [46]:

csv_folder = "/Users/mariem/Desktop/Project_ADA/newyorker_caption_contest/data"

def update_metadata_from_csv(df, base_path=csv_folder):
    updated = []
    missing = []
    errors = []

    for i, row in df.iterrows():
        # Get metadata safely and ensure it's a dict
        meta = row.get("metadata")
        if not isinstance(meta, dict):
            meta = {}  # create a dict if metadata is None or not a dict

        # check if update needed

        need_caps = ("num_captions" not in meta) or (meta.get("num_captions") in [None, ""])
        need_votes = ("num_votes" not in meta) or (meta.get("num_votes") in [None, ""])
        if not (need_caps or need_votes):
            # still write back in case metadata was None -> dict (optional)
            if row.get("metadata") is None:
                df.at[i, "metadata"] = meta
            continue

        # Build expected CSV path (basename(data) or fallback contest_id.csv)
        data_field = row.get("data") or ""
        #candidate1 = os.path.join(base_path, os.path.basename(data_field)) if data_field else None
        candidate2 = os.path.join(base_path, f"{int(row['contest_id'])}.csv") if pd.notna(row.get("contest_id")) else None

        csv_path = None
        for cand in (candidate1, candidate2):
            if cand and os.path.exists(cand):
                csv_path = cand
                break

        if csv_path is None:
            missing.append({"contest_id": row.get("contest_id"), "tried": [candidate1, candidate2]})
            # ensure metadata cell is at least a dict (so later code won't break)
            df.at[i, "metadata"] = meta
            continue

        # Read CSV and compute values (assume votes exists)
        try:
            cdf = pd.read_csv(csv_path)
            num_captions = int(len(cdf))
            if "votes" not in cdf.columns:
                raise KeyError(f"'votes' column not found in {csv_path}")
            num_votes = int(cdf["votes"].sum(skipna=True))

            # update dict and write back to dataframe
            meta["num_captions"] = num_captions
            meta["num_votes"] = num_votes
            df.at[i, "metadata"] = meta  # important: write back into the dataframe

            updated.append({"contest_id": row.get("contest_id"), "csv_path": csv_path, "num_captions": num_captions, "num_votes": num_votes})
        except Exception as e:
            errors.append({"contest_id": row.get("contest_id"), "csv_path": csv_path, "error": str(e)})
            # keep metadata as dict even if we failed
            df.at[i, "metadata"] = meta

    
    return df, updated, missing, errors

# Run it
df_contests, updated, missing, errors = update_metadata_from_csv(df_contests)



In [44]:
df_contests.iloc[250:300]

,contest_id,image,data,metadata
250,762,images/762.jpg,data/762.csv,"{'num_captions': 7076, 'num_votes': 901611, 'i..."
251,763,images/763.jpg,data/763.csv,"{'num_captions': 9007, 'num_votes': 889556, 'i..."
252,764,images/764.jpg,data/764.csv,"{'num_captions': 6310, 'num_votes': 665368}"
253,765,images/765.jpg,data/765.csv,"{'num_captions': 6932, 'num_votes': 763962}"
254,766,images/766.jpg,data/766.csv,"{'num_captions': 5608, 'num_votes': 731238}"
255,767,images/767.jpg,data/767.csv,"{'num_captions': 7912, 'num_votes': 768653}"
256,768,images/768.jpg,data/768.csv,"{'num_captions': 6526, 'num_votes': 779519}"
257,769,images/769.jpg,data/769.csv,"{'num_captions': 8039, 'num_votes': 921031}"
258,770,images/770.jpg,data/770.csv,"{'num_captions': 5353, 'num_votes': 531879}"
259,771,images/771.jpg,data/771.csv,"{'num_captions': 6861, 'num_votes': 616562}"
